# A project's Trino reads the catalogs of its Data Providers

Run from a notebook of the TRE (`cvd-2026`), signed in as a member of the project. It checks, one step at a time, that Trino reads the catalogs of the
Data Providers with the member's own token, and says which step fails, so that the failure says where to look:

1. the token that the notebook holds names the project, each Data Provider as an audience, and the second factor;
2. Trino takes the token, and has a catalog for each Data Provider;
3. Trino lists the tables of each Data Provider's `released` namespace, which is the first call that it makes to the Data Provider's Polaris, with the member's token and
   not with the placeholder with which the catalog starts: a refusal here (401 or 403) means Trino still calls Polaris with the placeholder;
4. Trino reads the table that each Data Provider has released, and gets its columns;
5. Trino is refused a table that a Data Provider has not released.

Needs two tables on each Data Provider, which its own smoke test writes (`afon-cli.sh manage-project run <data project> spark-polaris-smoke`): `cvd.staging.smoke_test`,
which the Data Provider then releases (`afon-cli.sh manage-project run <data project> manage-project-grants release cvd smoke_test`), and `cvd.staging.smoke_test_unreleased`,
which stays in `staging`. Change the cell below for other projects, Data Providers or tables.

In [ ]:
# What is read, and where: the Trino catalog of each Data Provider (its id in the TRE's config, in lower case) and the audience of its tokens
PROJECT      = "cvd-2026"
CATALOGS     = {"lon_cvd": "provider-lon", "nw_cvd": "provider-nw"}
RELEASED     = "released"                    # the namespace that the project may read
NOT_RELEASED = "staging"                     # the namespace that it may not
TABLE        = "smoke_test"                  # released by each Data Provider
UNRELEASED_TABLE = "smoke_test_unreleased"   # made in NOT_RELEASED, and not released: it must exist there
TRINO_HOST   = "trino"
TRINO_PORT   = 8443

results = {}

def record(step, ok, detail=""):
    """ok is True (pass), False (fail), or None: not conclusive, which the result says but does not count as a failure"""
    results[step] = ok
    label = "PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE"
    print(label, "-", step, ("- " + detail) if detail else "")

## 1. The token

`OIDC_TOKEN` is put in the environment of the notebook when it starts, from the member's sign-in. Only the claims are printed, never the token.

In [ ]:
import os, json, base64, time

token = os.environ.get("OIDC_TOKEN")
claims = {}
if not token:
    record("token is in the environment", False, "OIDC_TOKEN is not set: sign out and in again, or the hub does not pass it on")
else:
    payload = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
    for name in ("iss", "aud", "azp", "acr", "project", "polaris_principal", "email"):
        print(f"{name} = {claims.get(name)}")
    remaining = claims["exp"] - time.time()
    record("token names the project", PROJECT in (claims.get("project") or []), f"project = {claims.get('project')}")
    for catalog, audience in CATALOGS.items():
        record(f"token is for the Data Provider of {catalog}", audience in (claims.get("aud") or []), f"aud = {claims.get('aud')}")
    record("token says a second factor was used", claims.get("acr") == "mfa", f"acr = {claims.get('acr')}")
    record("token has not expired", remaining > 0, f"{remaining / 3600:.1f} hours left")

## 2. Trino takes the token and has the catalogs

The token is both how the member authenticates to Trino (as its access token) and what Trino passes on to the Data Provider's Polaris (as the extra credential
`token`). Trino runs queries as the user that the token names, `polaris_principal`, and no other.

In [ ]:
from trino.auth import JWTAuthentication
from trino.dbapi import connect

CA = os.environ.get("TRINO_CA_CERT", "/etc/trino-tls/ca.crt")   # the coordinator's certificate, which is self-signed

def cursor():
    return connect(host=TRINO_HOST, port=TRINO_PORT, http_scheme="https", verify=CA, user=claims.get("polaris_principal"),
                   auth=JWTAuthentication(token), extra_credential=[("token", token)]).cursor()

def run(sql):
    cur = cursor()
    cur.execute(sql)
    return cur.fetchall()

STEP = "Trino takes the token"
try:
    catalogs = {row[0] for row in run("SHOW CATALOGS")}
    record(STEP, True, f"{len(catalogs)} catalogs")
    for catalog in CATALOGS:
        record(f"Trino has the catalog {catalog}", catalog in catalogs, "" if catalog in catalogs else f"it has {sorted(catalogs)}")
except Exception as e:
    record(STEP, False, f"{type(e).__name__}: {e}")

## 3. Trino lists the schemas of each Data Provider

The first call that Trino makes to the Data Provider's Polaris. It is the Data Provider that answers: a 401 or 403 is its refusal of the token that Trino sent.

In [ ]:
def classify(e):
    text = f"{type(e).__name__}: {e}"
    refused = any(word in text.lower() for word in ("401", "403", "unauthorized", "forbidden", "not authorized", "access denied",
                                                                       "cannot obtain metadata", "failed to load table"))
    # Trino's wording of what a catalog answers, for which the proxy's log has the status: a refusal is the cause when the token is bad,
    # and a table that is not there is worded "does not exist", which is not a refusal
    return refused, text

for catalog in CATALOGS:
    step = f"Trino lists the tables of {catalog}.{RELEASED}"
    try:
        tables = [row[0] for row in run(f"SHOW TABLES FROM {catalog}.{RELEASED}")]
        record(step, True, f"tables: {tables}")
    except Exception as e:
        refused, text = classify(e)
        record(step, False, ("the Data Provider refused what Trino sent (the placeholder, not the member's token?): " if refused else "") + text[:400])

## 4. Trino reads the table that each Data Provider released

In [ ]:
for catalog in CATALOGS:
    step = f"Trino reads {catalog}.{RELEASED}.{TABLE}"
    try:
        cur = cursor()
        cur.execute(f"SELECT * FROM {catalog}.{RELEASED}.{TABLE} LIMIT 5")
        rows = cur.fetchall()
        columns = [d[0] for d in cur.description]
        if rows:
            record(step, True, f"columns {columns}, {len(rows)} rows, read from the Data Provider's storage with the credentials that its Polaris vends")
        else:
            record(step, None, f"columns {columns}, but no rows, so no data file was read: has the Data Provider written the table (spark-polaris-smoke)?")
    except Exception as e:
        refused, text = classify(e)
        if "does not exist" in text.lower() or "not found" in text.lower():
            record(step, None, f"the Data Provider has not released {TABLE}: {text[:300]}")
        else:
            record(step, False, ("refused: " if refused else "") + text[:400])

## 5. Trino is refused a table that has not been released

The member may read only the `released` namespace. The table must exist in `staging`, or the refusal is only a missing table, which proves nothing.

In [ ]:
for catalog in CATALOGS:
    step = f"Trino is refused {catalog}.{NOT_RELEASED}.{UNRELEASED_TABLE}"
    try:
        run(f"SELECT * FROM {catalog}.{NOT_RELEASED}.{UNRELEASED_TABLE} LIMIT 1")
        record(step, False, "the query was answered: the project can read what the Data Provider has not released")
    except Exception as e:
        refused, text = classify(e)
        if "does not exist" in text.lower() or "not found" in text.lower():
            record(step, None, f"the table does not exist in {NOT_RELEASED}, so there was nothing to refuse: {text[:300]}")
        elif refused:
            record(step, True, text[:300])
        else:
            record(step, None, f"refused, but not clearly for lack of access (does the table exist in {NOT_RELEASED}?): {text[:300]}")

## Result

In [ ]:
print()
for step, ok in results.items():
    print("PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE", "-", step)
print()
if any(ok is False for ok in results.values()):
    print("SOMETHING FAILED: the first FAIL above says where")
elif any(ok is None for ok in results.values()):
    print("NO FAILURE, BUT NOT EVERYTHING WAS CONCLUDED: see INCONCLUSIVE")
else:
    print("ALL PASS")